# Knox Design Analysis Notebook

Spring 2026

EC552 - Computational Synthetic Biology for Engineers

Team Members Names: Varsha Athreya & Melissa Regalado 

# Notes From James

Depending on number of rules, the rule evaluation algorithm can take 10-30 minutes to run. Rule Evaluations are saved in Neo4j, so you do not
need to rerun the rule evaluation algorithm, call getRuleEvaluation with the name of the evaluation (only 2-5 seconds to retrieve).

Run rule evaluations with:
- ruleEvaluateByGroup
- ruleEvaluateByDesigns


Get rule evaluations with
- getRuleEvaluation


Delete rule evaluations with
- deleteRuleEvaluation


This notebook is to serve as a starting point for doing your analysis.

Feel free to change this notebook in any way you see fit.


# Imports

In [13]:
import requests
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from sklearn import tree
from sklearn.model_selection import train_test_split

# Knox Request Functions

In [ ]:
url = 'http://localhost:8080'

def ruleEvaluateByGroup(evalName, groupID, ruleGroupID, labelingMethod="sign"):
    """
    API Request to Knox to run Rule Evaluation Algorithm.
    
    Returns:
    metrics (list of pandas DataFrames): purity_metrics, designToRule
    """
    response = requests.post(url + '/rule/evaluate?' + 
                             "evaluationName=" + evalName + '&' + 
                             "designGroupID=" + groupID + '&' +
                             "rulesGroupID=" + ruleGroupID + '&' +
                             "labelingMethod=" + labelingMethod
    )

    return processRuleEval(response)



def ruleEvaluateByDesigns(evalName, designIDs, ruleGroupID, designScores, labelingMethod="sign"):
    designSpaceIDs = listToStringList(designIDs)
    designScoresStr = listToStringList(designScores) # designScores should be a list of strings, may have to convert floats to strings before running

    # Submit Request
    response = requests.post(url + '/rule/evaluate?' + 
                             "evaluationName=" + evalName + '&' + 
                             "designSpaceIDs=" + designSpaceIDs + '&' + 
                             "rulesGroupID=" + ruleGroupID + '&' + 
                             "designScores=" + designScoresStr + '&' + 
                             "labelingMethod=" + labelingMethod
    )

    return processRuleEval(response)


def getRuleEvaluation(evalName):
    response = requests.get(url + '/rule/getEvaluation?' + "evaluationName=" + evalName)

    return processRuleEval(response)


def deleteRuleEvaluation(evalName):
    response = requests.delete(url + '/rule?' + "evaluationName=" + evalName)

    if not response.text:
        return f'"{evalName}" Sucessfully Deleted'
    else:
        return response.text
    

def processRuleEval(response):
    # Change to Pandas DataFrame
    json_data = json.loads(response.text)

    purity_metrics_df = pd.DataFrame(json_data["evaluationResults"]).T

    designToRule_df = pd.DataFrame(json_data["designToRule"], index=json_data["designToRule"]["designIDs"])
    cols = designToRule_df.columns.to_list()
    cols.remove("labels")
    cols.remove("scores")
    cols.remove("designIDs")
    cols = ["labels", "scores"] + cols
    designToRule_df = designToRule_df[cols]

    return purity_metrics_df.sort_values("Impact"), designToRule_df.sort_values("scores")


def listToStringList(list_input):
    return ",".join(list_input)
    


# Decision Tree Functions

In [ ]:
def exampleTree(X, y, **kwargs):
    # Mess around with different parameters
    
    dt_clf = tree.DecisionTreeClassifier(
        splitter=kwargs.get('splitter', 'best'),
        max_depth=kwargs.get('max_depth', None),
        min_samples_split=kwargs.get('min_samples_split', 400),
        min_samples_leaf=kwargs.get('min_samples_leaf', 200),
        max_features=kwargs.get('max_features', None),
        max_leaf_nodes=kwargs.get('max_leaf_nodes', None)
    )
    
    dt_clf = dt_clf.fit(X, y)

    return dt_clf

## Build more Trees, multiclassification and regression


# Design Analysis

### Run RuleEvaluation

In [ ]:
#deleteRuleEvaluation('test')

In [ ]:
# Use this for (gnn_predicted_scores.csv) designs already have the attached scores

evalName = 'test'
groupID = 'group1'
ruleGroupID = 'ruleGroup'

labelingMethod = 'median'

purity_metrics_df, designToRule_df = ruleEvaluateByGroup(evalName, groupID, ruleGroupID, labelingMethod)

In [ ]:
# Use this for extra credit 1 (true_scores.csv & transformer_predicted_scores.csv) attach new scores to the designs

evalName = "test"
designIDs = [f'outputPrefix_design_{i}' for i in range(1, 3997)]
designScores = [str(j) for j in pd.read_csv(r"C:\Path\to\transformer_predicted_scores.csv")["weight"].to_list()]
ruleGroupID = 'ruleGroup'

labelingMethod = 'median'

purity_metrics_df, designToRule_df = ruleEvaluateByDesigns(evalName, designIDs, ruleGroupID, designScores, labelingMethod)

### View DataFrames

In [ ]:
purity_metrics_df

In [ ]:
designToRule_df

## Extract Features and Labels from designToRule_df

In [ ]:
feature_names = designToRule_df.columns.to_list()[2:]
X = designToRule_df.iloc[:, 2:].to_numpy(dtype=int)
y_labels = designToRule_df["labels"].to_numpy(dtype=int)
y_scores = designToRule_df["scores"].to_numpy(dtype=float)


## Train - Test Split

## Build Trees

## Plot Trees

## Save Trees

## Save Purity Metrics and DesignToRule DF to CSV

# Deliverables